# Dataloader test: memmap enumeration vs DataLoader

Reproduces the slow loading behavior when using `decimate_factor > 1`: the dataloader reads **full** segments from disk then decimates in memory, so I/O is ~decimate_factor× heavier.

**Config:** Set `PREBUILT_MMAP_DIR` to your per-split memmap (e.g. `subseqs_mmap_all`). Use `decimate_factor=10` to mimic the decimated run; use `decimate_factor=1` (or pre-decimated data) for faster loading.

In [ ]:
import time
from pathlib import Path

import numpy as np

# ── Config (match run_tcn_baseline_ecei_mc_decimated.sh) ──
PREBUILT_MMAP_DIR = "/path/to/subseqs_mmap_all"  # e.g. .../scratch/soen_fusion_zero/subseqs_mmap_all
DECIMATE_FACTOR = 10  # 1 = no decimation (fast). 10 = same as decimated run (10× more I/O per sample)
SPLIT = "train"  # which split to enumerate

## 1. Enumerate all data in the memmap (raw read)

Opens the per-split RaggedMmaps and iterates over every sequence, reading X/target/weight with the same logic as `PrebuiltPerSplitSubseqDataset.__getitem__` (including decimation). Forces actual disk read by touching every element (e.g. `.sum()`). This replicates the **minimum** cost of loading the split.

In [ ]:
from mmap_ninja import RaggedMmap

root = Path(PREBUILT_MMAP_DIR)
sub = root / SPLIT
if not (sub / "X").exists():
    raise FileNotFoundError(f"Expected {sub / 'X'} (per-split layout)")

X_mmap = RaggedMmap(str(sub / "X"))
target_mmap = RaggedMmap(str(sub / "target"))
weight_mmap = RaggedMmap(str(sub / "weight"))
labels = np.load(sub / "labels.npy")
n = len(labels)
print(f"Split '{SPLIT}': {n} sequences, decimate_factor={DECIMATE_FACTOR}")

t0 = time.perf_counter()
bytes_read = 0
for i in range(n):
    X = np.ascontiguousarray(X_mmap[i]).astype(np.float32)
    target = np.asarray(target_mmap[i], dtype=np.float32).copy()
    weight = np.asarray(weight_mmap[i], dtype=np.float32).copy()
    if DECIMATE_FACTOR > 1:
        X = X[..., ::DECIMATE_FACTOR].copy()
        target = target[::DECIMATE_FACTOR].copy()
        weight = weight[::DECIMATE_FACTOR].copy()
    # Force actual read (avoid lazy/cached)
    bytes_read += X.nbytes + target.nbytes + weight.nbytes
    _ = X.sum() + target.sum() + weight.sum()

elapsed = time.perf_counter() - t0
print(f"  Enumerated {n} samples in {elapsed:.2f} s  ({n / elapsed:.1f} samples/s)")
print(f"  Effective data touched (after decimation): {bytes_read / 1e9:.2f} GB  ({bytes_read / 1e6 / elapsed:.1f} MB/s)")
if DECIMATE_FACTOR > 1:
    print(f"  (With decimate_factor={DECIMATE_FACTOR}, full segments are read from disk first, so actual I/O is ~{DECIMATE_FACTOR}× larger.)")

## 2. Reimplement the dataloader

Same dataset logic as `PrebuiltPerSplitSubseqDataset` and a PyTorch `DataLoader` (batch_size, num_workers, pin_memory). Enumerate one epoch to replicate training-time load cost.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class PrebuiltPerSplitSubseqDataset(Dataset):
    """Mirror of disruptcnn.dataset_original.PrebuiltPerSplitSubseqDataset for local testing."""

    def __init__(self, root, decimate_factor=1, split="train"):
        self._root = Path(root)
        self._decimate = max(1, int(decimate_factor))
        self._split = split
        sub = self._root / split
        self._X = RaggedMmap(str(sub / "X"))
        self._target = RaggedMmap(str(sub / "target"))
        self._weight = RaggedMmap(str(sub / "weight"))
        self._labels = np.load(sub / "labels.npy")

    def __len__(self):
        return len(self._labels)

    def __getitem__(self, index):
        X = np.ascontiguousarray(self._X[index]).astype(np.float32)
        target = np.asarray(self._target[index], dtype=np.float32).copy()
        weight = np.asarray(self._weight[index], dtype=np.float32).copy()
        if self._decimate > 1:
            X = X[..., ::self._decimate].copy()
            target = target[::self._decimate].copy()
            weight = weight[::self._decimate].copy()
        return (
            torch.from_numpy(X),
            torch.from_numpy(target),
            torch.from_numpy(weight),
        )


# Dataloader settings (match training: batch 16, num_workers 4)
BATCH_SIZE = 16
NUM_WORKERS = 4  # 0 = main process only (easier to profile)

ds = PrebuiltPerSplitSubseqDataset(PREBUILT_MMAP_DIR, decimate_factor=DECIMATE_FACTOR, split=SPLIT)
loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

n_batches = len(loader)
print(f"DataLoader: {len(ds)} samples, {n_batches} batches (batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS})")

t0 = time.perf_counter()
n_samples = 0
for batch in loader:
    X, target, weight = batch
    n_samples += X.shape[0]
    _ = X.sum().item() + target.sum().item() + weight.sum().item()  # force materialization

elapsed = time.perf_counter() - t0
print(f"  One epoch: {elapsed:.2f} s  ({n_samples / elapsed:.1f} samples/s)")
print(f"  Batches/s: {n_batches / elapsed:.1f}")